In [18]:
#Statistical arbitrage with Cointegration
# Pairs Trading & Statistical Arbitrage

#Statistical arbitrage refers to strategies that employ some statistical model or method to take advantage of what 
# appears to be relative mispricing of assets, while maintaining a level of market neutrality.

#Pairs trading is a conceptually straightforward strategy that has been employed by algorithmic 
# traders since at least the mid-eighties (Gatev, Goetzmann, and Rouwenhorst 2006). 
# The goal is to find two assets whose prices have historically moved together, 
# track the spread (the difference between their prices), and, once the spread widens, 
# buy the loser that has dropped below the common trend and short the winner. If the relationship persists, 
# the long and/or the short leg will deliver profits as prices converge and the positions are closed.

#This approach extends to a multivariate context by forming baskets from multiple securities 
# and trading one asset against a basket of two baskets against each other.



In [19]:
#Pairs Trading in Practice

#In practice, the strategy requires two steps:

#Formation phase: Identify securities that have a long-term mean-reverting relationship. 
# Ideally, the spread should have a high variance to allow for frequent profitable trades 
# while reliably reverting to the common trend.
#Trading phase: Trigger entry and exit trading rules as price movements cause thespread to diverge and converge.
#Several approaches to the formation and trading phases have emerged from 
# increasingly active research in this area, across multiple asset classes, over the last several years. 
# The book outlines the key differences between them; the notebook dives into an example application.

In [20]:
import warnings
warnings.filterwarnings('ignore')

In [21]:
from collections import Counter

from time import time
from pathlib import Path

import numpy as np
import pandas as pd

from pykalman import KalmanFilter
from statsmodels.tsa.stattools import coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
idx = pd.IndexSlice
sns.set_style('whitegrid')

In [23]:
def format_time(t):
    m_, s = divmod(t, 60)
    h, m = divmod(m_, 60)
    return f'{h:>02.0f}:{m:>02.0f}:{s:>02.0f}'

In [24]:
#Johansen Test Critical Values
critical_values = {0: {.9: 13.4294, .95: 15.4943, .99: 19.9349},
                   1: {.9: 2.7055, .95: 3.8415, .99: 6.6349}}
    
trace0_cv = critical_values[0][.95] # critical value for 0 cointegration relationships
trace1_cv = critical_values[1][.95] # critical value for 1 cointegration relationship

In [25]:
STORE = 'Users/Massimiliano/assets1.h5'

In [29]:
def get_backtest_prices():
    with pd.HDFStore('assets1.h5') as store:
        tickers = store['tickers']

    with pd.HDFStore(STORE) as store:
        prices = (pd.concat([
            store['stooq/us/nyse/stocks/prices'],
            store['stooq/us/nyse/etfs/prices'],
            store['stooq/us/nasdaq/etfs/prices'],
            store['stooq/us/nasdaq/stocks/prices']])
                  .sort_index()
                  .loc[idx[tickers.index, '2016':'2019'], :])
    print(prices.info(null_counts=True))
    prices.to_hdf('backtest.h5', 'prices')
    tickers.to_hdf('backtest.h5', 'tickers')

In [30]:
get_backtest_prices()

KeyError: 'No object named tickers in the file'